# Exploratory Data Analysis

This notebook explores the raw KMT dataset published by Nnamoko et al. (2022). 88 participants typed fictitious card details into a GUI app that resembles an online payment form, while their keyboard and mouse events were logged.

First, I look at the structure of the dataset and load a sample to get familiar with the data format. Then I load the whole dataset to look for data-quality issues and potentially discriminating features.

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [2]:
raw_kmt_dataset = Path(os.environ.get("KMT_DATA_DIR", "../data/raw_kmt_dataset"))
json_file_list = sorted(path.name for path in raw_kmt_dataset.glob("raw_kmt_user_*.json"))
if not json_file_list:
    raise FileNotFoundError(f"No KMT JSON files in {raw_kmt_dataset}")
json_file_list[:5]


['raw_kmt_user_0001.json',
 'raw_kmt_user_0002.json',
 'raw_kmt_user_0003.json',
 'raw_kmt_user_0004.json',
 'raw_kmt_user_0005.json']

The `raw_kmt_dataset` directory contains one JSON file per user (88 files in total). Let's look at the JSON structure of the first one.

In [3]:
with open(raw_kmt_dataset / json_file_list[0], 'r') as f:
    user1 = json.load(f)

print(user1.keys())

dict_keys(['details', 'true_data', 'false_data'])


In [4]:
print(user1["details"])
print(user1["false_data"].keys())
print(user1["true_data"].keys())

{'ID': 'CDID0001', 'Provider': 'Discover', 'Name': 'Mrs Ellie Miller', 'Card Number': 4422891459068728, 'CVC': 646, 'Expiry': '09/23'}
dict_keys(['test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'test_6', 'test_7', 'test_8', 'test_9', 'test_10'])
dict_keys(['test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'test_6', 'test_7', 'test_8', 'test_9', 'test_10'])


In [5]:
print(user1["true_data"]["test_1"].keys())

dict_keys(['key_events', 'mouse_events'])


In [6]:
user1_key = pd.DataFrame(user1["true_data"]["test_1"]["key_events"])

display(user1_key)
display(user1_key.describe(include="all"))
display(user1_key.dtypes)

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
0,shift,pressed,Name,True,2022-02-06 18:57:09.981740,1644173829.9817405
1,m,pressed,Name,True,2022-02-06 18:57:10.121021,1644173830.1210213
2,shift,released,Name,True,2022-02-06 18:57:10.169470,1644173830.1694703
3,m,released,Name,True,2022-02-06 18:57:10.218179,1644173830.21818
4,r,pressed,Name,True,2022-02-06 18:57:10.293582,1644173830.293582
...,...,...,...,...,...,...
113,tab,released,Exp y,True,2022-02-06 18:57:26.707177,1644173846.7071774
114,2,pressed,Exp y,True,2022-02-06 18:57:26.834916,1644173846.8349168
115,2,released,Exp y,True,2022-02-06 18:57:26.895162,1644173846.8951626
116,3,pressed,Exp y,True,2022-02-06 18:57:26.958561,1644173846.9585614


,Key,Event,Input Box,Text Changed,Timestamp,Epoch
count,118,118,118,118,118,118
unique,21,2,5,2,116,116
top,backspace,released,Name,True,2022-02-06 18:57:10.756190,1644173830.7561908
freq,14,61,64,112,2,2


Key              str
Event            str
Input Box        str
Text Changed    bool
Timestamp        str
Epoch            str
dtype: object

Each file has three top-level keys:
- `details`: the fictitious card details assigned to the user (provider, name, card number, CVC and expiry date);
- `true_data`: 10 **genuine** attempts (`test_1`–`test_10`), in which the user typed their own assigned card details;
- `false_data`: 10 **impostor** attempts, in which other participants typed this user's card details, each of them once.

Each attempt contains two lists of events: `key_events` and `mouse_events`.

## Keyboard data
The keyboard data was collected at the event level: one row per key press or release.

Six features were registered:
- `Key`: the name of the key, e.g. `m`, `shift` or `numpad4`;
- `Event`: the event type, either `pressed` or `released`;
- `Input Box`: the text field of the GUI app the user is typing in (`Name`, `Card No`, `CVC`, `Exp m` or `Exp y`);
- `Text Changed`: whether the event changed the text in the field, e.g. typing a fourth digit into the three-digit CVC field doesn't change it;
- `Timestamp`: date and time with microsecond precision;
- `Epoch`: seconds elapsed since 1970-01-01 00:00:00 UTC (Unix time), stored as a string.

`Timestamp` and `Epoch` carry the same information (apart from rare differences of about a millisecond), so one of them can be dropped.

**NOTE**<br>
- Modifier keys such as `shift` are held for a substantially longer time than alphanumeric keys. They may distort the average dwell time, so consider dropping them or treating them separately. On the other hand, their hold time may be useful in itself.
- Whether a user moves between the form fields with `tab` and/or `left click` may have discriminating power.

Let's now look at the whole concatenated keyboard event dataset to check for other possible discriminating features and potential issues with the data.

In [7]:
dfs = []
for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for no in range(1, 11):
            dfs.append(pd.DataFrame(user_data["true_data"][f"test_{no}"]["key_events"]))
            dfs.append(pd.DataFrame(user_data["false_data"][f"test_{no}"]["key_events"]))

big_keyboard_df = pd.concat(dfs).astype({
    "Epoch": "Float64"
})

display(big_keyboard_df.describe(include="all"))
display(big_keyboard_df.isna().sum())

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
count,172563,172563,172563,172563,172563,172563.0
unique,82,2,6,2,171463,<NA>
top,backspace,pressed,Name,True,2022-03-01 13:48:50.519671,<NA>
freq,7925,87608,79905,162836,4,<NA>
mean,NaN,NaN,NaN,NaN,NaN,1647302420.65725
std,NaN,NaN,NaN,NaN,NaN,1935228.31744
min,NaN,NaN,NaN,NaN,NaN,1643113826.341281
25%,NaN,NaN,NaN,NaN,NaN,1646151579.757441
50%,NaN,NaN,NaN,NaN,NaN,1647353218.034949
75%,NaN,NaN,NaN,NaN,NaN,1648984188.621257


Key             0
Event           0
Input Box       0
Text Changed    0
Timestamp       0
Epoch           0
dtype: int64

- In total, 172,563 key events were recorded in the experiments.
- They involve 82 unique keys.
- Only `pressed` and `released` events were logged. No issues there so far.
- `Input Box` has a sixth value besides the five fields seen above: `Null`, which according to the paper means that no text field was in focus.
- There is no missing data in any of the features.

In [8]:
big_keyboard_df["Key"].unique()

<ArrowStringArray>
[          'shift',               'm',               'r',               's',
        'spacebar',               'w',               'e',       'backspace',
               'l',               'i',             'tab',               '4',
               '2',               '8',               '9',               '1',
               '5',               '0',               '6',               '7',
               '3',          'rshift',         'numpad4',         'numpad2',
         'numpad8',         'numpad9',         'numpad1',         'numpad5',
         'numpad0',         'numpad6',         'numpad7',         'numpad3',
        'capslock', 'numpadsubstract',               '.',               'j',
               ',',               'h',               'd',               't',
               'a',               'y',               'u',               'o',
         'numlock',               'f',           'enter',               'n',
               'g',               '-',               '`',

**NOTE**

Possible discriminating habits:
- using the numpad (`numpad0`–`numpad9`) or the top row for digits;
- moving between text fields with `tab` and/or a mouse `left click`;
- typing capitals with `shift` or `capslock`, and using the left or the right `shift`;
- making typing mistakes and correcting them with `backspace` and/or `delete`;
- navigating with the arrow keys.

In [9]:
big_keyboard_df["Event"].value_counts()

Event
pressed     87608
released    84955
Name: count, dtype: int64

**NOTE**<br>
There are 2,653 more `pressed` than `released` events, so not all key events come in press–release pairs. This implies that the dataset has missing events and/or that the logging has quirks.

Let's look a little deeper into that.

In [10]:
big_keyboard_df.groupby(["Key", "Event"]).count()

Input Box  Text Changed  Timestamp  Epoch
Key Event                                              
#   pressed          22            22         22     22
    released         22            22         22     22
'   pressed           1             1          1      1
    released          1             1          1      1
,   pressed          11            11         11     11
...                 ...           ...        ...    ...
x   released        106           106        106    106
y   pressed         613           613        613    613
    released        613           613        613    613
z   pressed          66            66         66     66
    released         66            66         66     66

[164 rows x 4 columns]

There are small differences for briefly pressed keys like `2` and `8`, and much bigger ones for `backspace`, `shift`, `rshift`, `capslock`, `spacebar` and more. For all of these keys there are more `pressed` than `released` events, which suggests that a key held down for a long time is logged as a series of `pressed` events.

However, something particularly interesting happens with the `tab` key: almost all of its events are `released` (3,188 `released` vs 19 `pressed`).

Let's look at some examples, starting with the `2` key, then `shift` and `tab`.

In [11]:
shifted_df = big_keyboard_df.groupby("Key", sort=False)[["Key","Event", "Epoch"]].shift(-1)
shifted_df.columns = ["Key_shifted", "Event_shifted", "Epoch_shifted"]
comparison_df = pd.concat([big_keyboard_df, shifted_df], axis=1)
comparison_df.loc[(comparison_df["Key"] == "2") & (comparison_df["Event"] == comparison_df["Event_shifted"])]
comparison_df[comparison_df["Epoch"].between(1648896881.0545242 - 5, 1648896881.0545242 + 30)]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch,Key_shifted,Event_shifted,Epoch_shifted
70,4,released,Card No,True,2022-04-02 10:54:36.271858,1648896876.271858,4,pressed,1649073768.692647
71,5,pressed,Card No,True,2022-04-02 10:54:36.418063,1648896876.418063,5,released,1648896876.577644
72,5,released,Card No,True,2022-04-02 10:54:36.577644,1648896876.577644,5,pressed,1648896885.870337
73,7,pressed,Card No,True,2022-04-02 10:54:37.261808,1648896877.261808,7,released,1648896877.469558
74,7,released,Card No,True,2022-04-02 10:54:37.469557,1648896877.469558,7,pressed,1648896897.060272
75,1,pressed,Card No,True,2022-04-02 10:54:37.592983,1648896877.592984,1,released,1648896877.759992
76,1,released,Card No,True,2022-04-02 10:54:37.759991,1648896877.759992,1,pressed,1648896886.395125
77,9,pressed,Card No,True,2022-04-02 10:54:40.169949,1648896880.16995,9,released,1648896880.319356
78,9,released,Card No,True,2022-04-02 10:54:40.319356,1648896880.319356,9,pressed,1649073779.429939
79,2,pressed,Card No,True,2022-04-02 10:54:40.796559,1648896880.797569,2,pressed,1648896881.054524


In one peculiar episode the `2` key was logged as pressed three times before its release, after which the user deleted two characters with `backspace`. The second press came 0.26 s after the first and the third only 0.02 s later. Perhaps the user held the key a little too long, so the operating system's auto-repeat typed two extra `2`s, or some app glitch happened and the user didn't see the result at first. I check the auto-repeat explanation in `features.ipynb`.

In [12]:
big_keyboard_df.loc[big_keyboard_df["Key"] == "shift"]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
0,shift,pressed,Name,True,2022-02-06 18:57:09.981740,1644173829.98174
2,shift,released,Name,True,2022-02-06 18:57:10.169470,1644173830.16947
9,shift,pressed,Name,True,2022-02-06 18:57:10.679335,1644173830.679336
15,shift,released,Name,True,2022-02-06 18:57:10.844363,1644173830.844363
20,shift,pressed,Name,True,2022-02-06 18:57:11.721636,1644173831.721636
...,...,...,...,...,...,...
26,shift,pressed,Name,True,2022-04-14 09:59:00.636272,1649930340.636272
27,shift,pressed,Name,True,2022-04-14 09:59:00.661282,1649930340.661283
28,shift,pressed,Name,True,2022-04-14 09:59:00.674279,1649930340.674279
29,shift,pressed,Name,True,2022-04-14 09:59:00.699285,1649930340.699286


Consecutive `shift` presses are easily explained by long presses, which the OS registers as consecutive `pressed` events (in the last rows above they come about 12–25 ms apart). `shift` is a modifier key, so it is often held down while other keys are typed.

In [13]:
big_keyboard_df.loc[big_keyboard_df["Key"] == "tab"]

,Key,Event,Input Box,Text Changed,Timestamp,Epoch
64,tab,released,Card No,True,2022-02-06 18:57:14.693870,1644173834.693871
101,tab,released,CVC,True,2022-02-06 18:57:24.125922,1644173844.125922
108,tab,released,Exp m,True,2022-02-06 18:57:25.602162,1644173845.602162
113,tab,released,Exp y,True,2022-02-06 18:57:26.707177,1644173846.707177
54,tab,released,Card No,True,2022-02-06 18:57:33.026100,1644173853.0261
...,...,...,...,...,...,...
32,tab,released,Card No,True,2022-04-05 09:42:20.966870,1649151740.966871
65,tab,released,CVC,True,2022-04-05 09:42:45.964544,1649151765.964545
72,tab,released,Exp m,True,2022-04-05 09:42:53.534079,1649151773.53408
77,tab,released,Exp y,True,2022-04-05 09:42:57.165202,1649151777.165202


For the `tab` key, I can't find a reasonable explanation just by looking at the data: its key-down is almost never logged, only its release.

The paper says that the events were logged with the Kivy Python library. In the notebook accompanying the paper, they drop `tab` events completely, explaining that Kivy "doesn't register tab releases, only presses", which is contrary to what I have found. Nevertheless, their two keyboard features, average `Dwell Time` (how long a key is held) and average `Flight Time` (time between a key release and the next key press), need press–release pairs, so unpaired `tab` events are useless for them. However, `tab` may still be worth keeping, as it may have discriminating value: some users prefer `tab` over a `left click` for jumping between fields.

**NOTE**<br>
Research typical `Kivy` issues and quirks.

## Mouse data
That's all for the first look at the keyboard data. Let's now look at the mouse events of one attempt: the first impostor attempt on user 1's card.

In [14]:
user1_mouse = pd.DataFrame(user1["false_data"]["test_1"]["mouse_events"]).astype({
    "Movement ID": "Int64",
    "Timestamp": "datetime64[us]",
    "Epoch": "Float64"
})

display(user1_mouse)
display(user1_mouse.describe(include='all'))
display(user1_mouse.dtypes)

,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
0,movement,"[386.0, 91.0]",2022-02-06 19:25:28.359473,1644175528.359473,1,NaN
1,left release,"[386.0, 91.00000000000001]",2022-02-06 19:25:28.408747,1644175528.408747,<NA>,NaN
2,movement,"[386.0, 92.0]",2022-02-06 19:25:28.421404,1644175528.421404,1,NaN
3,movement,"[387.0, 92.0]",2022-02-06 19:25:28.471485,1644175528.471485,1,NaN
4,movement,"[387.0, 93.0]",2022-02-06 19:25:28.496569,1644175528.496569,1,NaN
...,...,...,...,...,...,...
456,movement,"[373.0, 115.0]",2022-02-06 19:25:57.417238,1644175557.417238,11,NaN
457,movement,"[373.0, 113.0]",2022-02-06 19:25:57.443276,1644175557.443276,11,NaN
458,movement,"[374.0, 113.0]",2022-02-06 19:25:57.468304,1644175557.468304,11,NaN
459,left press,"[374.0, 113.00000000000001]",2022-02-06 19:25:57.517562,1644175557.517563,<NA>,NaN


,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
count,460,460,460,460.0,446.0,1.0
unique,3,449,NaN,<NA>,<NA>,NaN
top,movement,"[208.0, 94.0]",NaN,<NA>,<NA>,NaN
freq,446,4,NaN,<NA>,<NA>,NaN
mean,NaN,NaN,2022-02-06 19:25:41.622957,1644175541.622957,4.553812,0.0
min,NaN,NaN,2022-02-06 19:25:28.359473,1644175528.359473,1.0,0.0
25%,NaN,NaN,2022-02-06 19:25:31.510268,1644175531.510269,1.0,0.0
50%,NaN,NaN,2022-02-06 19:25:38.197915,1644175538.197916,2.0,0.0
75%,NaN,NaN,2022-02-06 19:25:52.593991,1644175552.593991,9.0,0.0
max,NaN,NaN,2022-02-06 19:25:57.517562,1644175557.517563,11.0,0.0


Event                      str
Coordinates             object
Timestamp       datetime64[us]
Epoch                  Float64
Movement ID              Int64
false_enters           float64
dtype: object

In [15]:
type(user1_mouse["Coordinates"].iloc[0])

list

Here is what I can tell from the first look at one attempt.

Each mouse event has the following fields:
- `Event`: the event type, `movement` for mouse movement and other values for clicks etc.;
- `Coordinates`: the `[X, Y]` cursor position on the GUI (parsed into the DataFrame as a `list`);
- `Timestamp` and `Epoch`: the time of the event (see the keyboard data above);
- `Movement ID`: a counter of mouse movements, restarting in each attempt. According to the paper, a new movement starts after 0.5 s without mouse movement;
- `false_enters`: I can't determine from the data what it represents, and the paper doesn't mention it (**NOTE** a strong candidate for dropping).

Similarly to the keyboard data, there are some issues with event logging. For starters, the second event is a `left release` with no `left press` before it. Click coordinates also carry floating-point noise, e.g. `91.00000000000001` instead of `91.0`.

Let's now load the whole mouse dataset, simply concatenating the data of each user.

In [16]:
dfs = []
for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for no in range(1, 11):
            dfs.append(pd.DataFrame(user_data["true_data"][f"test_{no}"]["mouse_events"]))
            dfs.append(pd.DataFrame(user_data["false_data"][f"test_{no}"]["mouse_events"]))

big_mouse_df = pd.concat(dfs)

display(big_mouse_df.isna().sum())

Event             1760
Coordinates       1760
Timestamp         1760
Epoch             1760
Movement ID      25320
false_enters    706984
dtype: int64

- There are exactly **1,760** `NA` values in each of the first four columns. The number corresponds to **88 users × 20 attempts**, i.e. one row per attempt.
- `false_enters` is missing in almost all rows (706,984), and `Movement ID` in 25,320 rows, which should correspond to events other than movements.

Let's make sure that the **1,760** `NA` values are in the same rows for each of the first four features, and see what values the remaining features take.

In [17]:
nas = big_mouse_df.loc[big_mouse_df["Event"].isna()]
print(nas.isna().sum())
nas.describe(include="all")

Event           1760
Coordinates     1760
Timestamp       1760
Epoch           1760
Movement ID     1760
false_enters       0
dtype: int64


,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
count,0,0,0,0,0.0,1760.000000
unique,0,0,0,0,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,0.278409
std,NaN,NaN,NaN,NaN,NaN,1.495793
min,NaN,NaN,NaN,NaN,NaN,0.000000
25%,NaN,NaN,NaN,NaN,NaN,0.000000
50%,NaN,NaN,NaN,NaN,NaN,0.000000
75%,NaN,NaN,NaN,NaN,NaN,0.000000


Only `false_enters` has values in these rows: each attempt's list of mouse events ends with one extra record that contains nothing else. The values range from 0 to 35, and most are 0 (mean 0.28). Since it is unknown what this feature represents, I will drop it in the loading function before more detailed analysis and feature engineering, together with these empty rows.

In [18]:
big_mouse_df.loc[big_mouse_df["Movement ID"].isna()]

,Event,Coordinates,Timestamp,Epoch,Movement ID,false_enters
55,left press,"[307.0, 383.00000000000006]",2022-02-06 18:57:08.782240,1644173828.7822409,NaN,NaN
56,left release,"[307.0, 383.00000000000006]",2022-02-06 18:57:08.856246,1644173828.8562467,NaN,NaN
74,left press,"[318.0, 312.0]",2022-02-06 18:57:09.122411,1644173829.1224115,NaN,NaN
78,left release,"[319.0, 305.0]",2022-02-06 18:57:09.186461,1644173829.186461,NaN,NaN
83,left press,"[324.0, 291.0]",2022-02-06 18:57:09.428796,1644173829.428797,NaN,NaN
...,...,...,...,...,...,...
160,left release,"[422.0, 380.99999999999994]",2022-04-05 09:41:55.375268,1649151715.375268,NaN,NaN
191,left press,"[288.0, 285.0]",2022-04-05 09:41:58.584616,1649151718.5846167,NaN,NaN
192,left release,"[288.0, 285.0]",2022-04-05 09:41:58.812721,1649151718.8127215,NaN,NaN
294,left press,"[376.0, 88.99999999999999]",2022-04-05 09:43:08.412875,1649151788.4128754,NaN,NaN


This confirms that missing `Movement ID` values correspond to events other than `movement` (plus the 1,760 trailing `false_enters` records).

In [19]:
big_mouse_df.drop(["false_enters"], axis=1, inplace=True)
big_mouse_df.dropna(subset=["Event"], inplace=True)

In [20]:
big_mouse_df.isna().sum()

Event              0
Coordinates        0
Timestamp          0
Epoch              0
Movement ID    23560
dtype: int64

In [21]:
big_mouse_df["Event"].unique()

<ArrowStringArray>
[           'movement',          'left press',        'left release',
    'scrolldown press',  'scrolldown release',      'scrollup press',
    'scrollup release',   'scrollright press', 'scrollright release',
         'right press',       'right release',    'scrollleft press',
  'scrollleft release',        'mouse4 press',      'mouse4 release']
Length: 15, dtype: str

Several types of mouse events were recorded:
- button clicks: `left press/release`, `right press/release` and `mouse4 press/release`;
- scrolling: `scrollup`, `scrolldown`, `scrollleft` and `scrollright` `press/release`;
- mouse movement: `movement`.

I am particularly curious about the distribution of the number of movements per attempt.

In [22]:
movement_counts = []

for file in json_file_list:
    with open(raw_kmt_dataset / file, 'r') as f:
        user_data = json.load(f)
        for test_no in range(1, 11):
            true_df = pd.DataFrame(user_data["true_data"][f"test_{test_no}"]["mouse_events"])
            movement_counts.append(true_df["Movement ID"].nunique())

            false_df = pd.DataFrame(user_data["false_data"][f"test_{test_no}"]["mouse_events"])
            movement_counts.append(false_df["Movement ID"].nunique())

movement_counts_series = pd.Series(movement_counts).rename("Movement Count")
movement_counts_series.agg([np.mean, np.std, np.min, np.max, np.median])

mean       7.007955
std        5.108810
min        1.000000
max       73.000000
median     7.000000
Name: Movement Count, dtype: float64

An attempt contains only a handful of mouse movements, so attempt-level
statistics aggregate relatively small samples.

The package loader now preserves nonempty mouse events, optional keyboard
fields, and event order. Undocumented `false_enters` trailer values are retained
as attempt metadata, not interpreted as model features. `Timestamp` is redundant
with the numeric `Epoch`, which is retained for interval calculations.

Floating-point precision is finer than the approximately 12 ms logging tick;
that sampling granularity, rather than the printed timestamp precision, limits
what can be inferred from short intervals.


In [23]:
from behavioral_ml.etl import load_data
mouse_df, key_df = load_data(raw_kmt_dataset)
assert len(mouse_df) == 706984
mouse_df.head()


Event  ...              Epoch
User ID Is Genuine Test ID Event ID Movement ID            ...                   
1       True       1       0        1            movement  ...  1644173827.568144
                           1        1            movement  ...  1644173827.580155
                           2        1            movement  ...  1644173827.592166
                           3        1            movement  ...  1644173827.605178
                           4        1            movement  ...   1644173827.61819

[5 rows x 3 columns]